# 02 — DistilBERT fine-tuning

**Owner:** Van (Modeler)  
**MLflow experiment:** `sentiment-distilbert`  
**Data:** `data/local/gold/gold_50k_training.csv` (gitignored; built by the gold EDA notebook)

Offline experimentation notebook — fine-tunes DistilBERT and compares it against the
sklearn baseline from `01_tfidf_logreg_tuning.ipynb`. Best model is registered as
`sentiment-distilbert` in the MLflow Registry.

Experiments: A baseline fine-tune → B weighted loss → C weighted loss + neutral oversample →
threshold tuning → final eval on test + OOT → register → (optional) LR sensitivity probes.

> **Dependencies:** `pip install datasets` if missing (transformers, torch are in `models/requirements.txt`).
>
> **Hardware warning:** this venv has CPU-only torch. 4 epochs × ~34K rows on CPU will take
> many hours. Strongly consider a GPU runtime (e.g. Colab — upload the CSV) or, for a quick
> wiring check, subsample `train_df` / set `EPOCHS = 1` first.

In [1]:
import torch
print(torch.cuda.is_available())       # True
print(torch.cuda.get_device_name(0))   # NVIDIA GeForce RTX 5080 Laptop GPU


True
NVIDIA GeForce RTX 5080 Laptop GPU


## [0] Imports and config

In [1]:
import subprocess, sys

# pyarrow on Windows can have broken DLL linkage after a partial install.
# Force-reinstall it first so datasets can import _dataset successfully.
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "--force-reinstall", "--no-cache-dir", "pyarrow"
])

_HEAVY_DEPS = ["datasets", "accelerate"]
_missing = []
for _pkg in _HEAVY_DEPS:
    try:
        __import__(_pkg)
    except ImportError:
        _missing.append(_pkg)

if _missing:
    print(f"Installing missing deps: {_missing}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + _missing)

print("All deps ready — restart the kernel now if pyarrow was just reinstalled.")

c:\SMU_MITB\MLE\CS611_MLE\social_listener_project\Machine-Learning-Engineering-Group-9-Social-Listener\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


All deps ready — restart the kernel now if pyarrow was just reinstalled.


In [2]:
from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
import mlflow
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import (
    classification_report,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from sklearn.utils import resample

# Resolve repo root whether the kernel cwd is repo root or notebooks/
ROOT = Path.cwd()
if not (ROOT / "data" / "local").exists() and (ROOT.parent / "data" / "local").exists():
    ROOT = ROOT.parent

DATA_PATH = ROOT / "data" / "local" / "gold" / "gold_50k_training.csv"
ARTIFACT_DIR = ROOT / "models" / "artifacts"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

LABELS = ["negative", "neutral", "positive"]
LABEL2ID = {l: i for i, l in enumerate(LABELS)}
ID2LABEL = {i: l for l, i in LABEL2ID.items()}
NEG_IDX = 0  # index of "negative" in LABELS

MODEL_CHECKPOINT = "distilbert-base-uncased"
MAX_LENGTH = 256    # sufficient for restaurant reviews; saves memory vs 512
BATCH_TRAIN = 16    # reduce to 8 if running on CPU
BATCH_EVAL = 32
EPOCHS = 4
LR = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
OUTPUT_DIR = str(ROOT / "checkpoints" / "distilbert-sentiment")
SEED = 42

USE_GPU = torch.cuda.is_available()
print(f"Repo root: {ROOT}")
print(f"Data file exists: {DATA_PATH.exists()}")
print(f"GPU available: {USE_GPU}")

Repo root: c:\SMU_MITB\MLE\CS611_MLE\social_listener_project\Machine-Learning-Engineering-Group-9-Social-Listener
Data file exists: True
GPU available: True


In [3]:
# MLflow: local Docker server first (host port 5001), file-store fallback.
# docker-compose maps host 5001 -> container 5000 (5000 is often taken on macOS).
import os
import requests

TRACKING_URI = "http://localhost:5001"
try:
    requests.get(TRACKING_URI, timeout=2)
    mlflow.set_tracking_uri(TRACKING_URI)
    # Docker is up — remap MinIO endpoint to localhost so host-side boto3 can reach it.
    os.environ["MLFLOW_S3_ENDPOINT_URL"] = "http://localhost:9000"
except requests.exceptions.RequestException:
    mlflow.set_tracking_uri(f"file:{(ROOT / 'mlruns').as_posix()}")
    # Docker is down — clear S3 vars so artifacts stay on local filesystem, not MinIO.
    os.environ.pop("MLFLOW_S3_ENDPOINT_URL", None)
    os.environ.pop("AWS_ACCESS_KEY_ID", None)
    os.environ.pop("AWS_SECRET_ACCESS_KEY", None)

mlflow.set_experiment("sentiment-distilbert")
print(f"MLflow tracking URI: {mlflow.get_tracking_uri()}")

MLflow tracking URI: http://localhost:5001


## [1] Data loading and splits

Identical shared split code to `01_tfidf_logreg_tuning.ipynb` — temporal OOT/demo cutoffs,
then random stratified 80/10/10 from the remaining pool. Demo CSVs are written by
notebook 01; the writes here are idempotent (same seeds, same output).

> Cutoffs adjusted to this export's actual range (2021-05-02 → 2022-04-10):
> rest 85.7% / OOT 10.2% / demo 4.1%.

In [4]:
df = pd.read_csv(DATA_PATH, parse_dates=["review_date"])
df = df.sort_values("review_date").reset_index(drop=True)

# ── Temporal holdouts ──────────────────────────────────────────────────────────
OOT_CUTOFF = "2021-12-11"
DEMO_CUTOFF = "2022-01-09"

demo_df = df[df.review_date >= DEMO_CUTOFF].copy()
oot_df = df[(df.review_date >= OOT_CUTOFF) & (df.review_date < DEMO_CUTOFF)].copy()
rest_df = df[df.review_date < OOT_CUTOFF].copy()

# ── Random splits from rest_df ─────────────────────────────────────────────────
rest_df = rest_df.sample(frac=1, random_state=SEED).reset_index(drop=True)

train_df, temp_df = train_test_split(
    rest_df, test_size=0.2, stratify=rest_df["label"], random_state=SEED
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.5, stratify=temp_df["label"], random_state=SEED
)

# ── Sanity check ───────────────────────────────────────────────────────────────
for name, split in [("train", train_df), ("val", val_df), ("test", test_df),
                    ("oot", oot_df), ("demo", demo_df)]:
    dist = split.label.value_counts(normalize=True).round(3).to_dict()
    print(f"{name:6s}  n={len(split):>6,}  {dist}")

y_train = train_df["label"].map(LABEL2ID).values
y_val = val_df["label"].map(LABEL2ID).values
y_test = test_df["label"].map(LABEL2ID).values
y_oot = oot_df["label"].map(LABEL2ID).values

train   n=34,413  {'positive': 0.716, 'negative': 0.212, 'neutral': 0.072}
val     n= 4,302  {'positive': 0.716, 'negative': 0.212, 'neutral': 0.072}
test    n= 4,302  {'positive': 0.716, 'negative': 0.212, 'neutral': 0.072}
oot     n= 5,147  {'positive': 0.704, 'negative': 0.217, 'neutral': 0.079}
demo    n= 2,056  {'positive': 0.739, 'negative': 0.198, 'neutral': 0.063}


In [5]:
def evaluate(y_true, y_pred, split_name: str, log_to_mlflow: bool = True) -> dict:
    """Compute and optionally log all metrics for a given split."""
    metrics = {
        f"{split_name}_f1_negative":        f1_score(y_true, y_pred, labels=[NEG_IDX], average="macro"),
        f"{split_name}_recall_negative":    recall_score(y_true, y_pred, labels=[NEG_IDX], average="macro"),
        f"{split_name}_precision_negative": precision_score(y_true, y_pred, labels=[NEG_IDX], average="macro"),
        f"{split_name}_f1_macro":           f1_score(y_true, y_pred, average="macro"),
        f"{split_name}_f1_neutral":         f1_score(y_true, y_pred, labels=[1], average="macro"),
        f"{split_name}_f1_positive":        f1_score(y_true, y_pred, labels=[2], average="macro"),
    }
    print(f"\n── {split_name.upper()} ──")
    print(classification_report(y_true, y_pred, target_names=LABELS))
    if log_to_mlflow:
        mlflow.log_metrics(metrics)
    return metrics

## [2] Tokenization

In [6]:
from datasets import Dataset
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)


def encode_df(frame: pd.DataFrame) -> Dataset:
    ds = Dataset.from_pandas(
        frame[["text", "label"]].assign(label=frame["label"].map(LABEL2ID)),
        preserve_index=False,
    )
    ds = ds.map(
        lambda b: tokenizer(b["text"], truncation=True,
                            padding="max_length", max_length=MAX_LENGTH),
        batched=True,
    )
    ds = ds.rename_column("label", "labels")
    ds.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
    return ds


train_ds = encode_df(train_df)
val_ds = encode_df(val_df)
test_ds = encode_df(test_df)
oot_ds = encode_df(oot_df)

print(train_ds)

Map: 100%|██████████| 5147/5147 [00:00<00:00, 9184.33 examples/s]

Dataset({
    features: ['text', 'labels', 'input_ids', 'attention_mask'],
    num_rows: 34413
})


from transformers import (
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
)


def get_training_args(run_name: str) -> TrainingArguments:
    return TrainingArguments(
        output_dir=f"{OUTPUT_DIR}/{run_name}",
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH_TRAIN,
        per_device_eval_batch_size=BATCH_EVAL,
        learning_rate=LR,
        weight_decay=WEIGHT_DECAY,
        warmup_ratio=WARMUP_RATIO,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="f1_negative",
        greater_is_better=True,
        fp16=USE_GPU,
        seed=SEED,
        report_to="none",
    )


def compute_hf_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "f1_negative": f1_score(labels, preds, labels=[NEG_IDX], average="macro"),
        "f1_macro":    f1_score(labels, preds, average="macro"),
    }


def run_distilbert_experiment(run_name: str, train_dataset, trainer_class=Trainer,
                              extra_params: dict = None):
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_CHECKPOINT, num_labels=3,
        id2label=ID2LABEL, label2id=LABEL2ID,
    )
    args = get_training_args(run_name)
    trainer = trainer_class(
        model=model, args=args,
        train_dataset=train_dataset,
        eval_dataset=val_ds,
        compute_metrics=compute_hf_metrics,
    )
    with mlflow.start_run(run_name=run_name):
        params = {"model": MODEL_CHECKPOINT, "epochs": EPOCHS, "lr": LR,
                  "max_length": MAX_LENGTH, "batch_size": BATCH_TRAIN}
        if extra_params:
            params.update(extra_params)
        mlflow.log_params(params)

        trainer.train()

        for split_name, ds, y_true in [
            ("val",  val_ds,  y_val),
            ("test", test_ds, y_test),
            ("oot",  oot_ds,  y_oot),
        ]:
            preds = np.argmax(trainer.predict(ds).predictions, axis=-1)
            evaluate(y_true, preds, split_name)

    return trainer


baseline_trainer = run_distilbert_experiment(
    "distilbert-baseline",
    train_ds,
    extra_params={"class_weight": "none"},
)

In [7]:
from transformers import (
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
)


def get_training_args(run_name: str) -> TrainingArguments:
    return TrainingArguments(
        output_dir=f"{OUTPUT_DIR}/{run_name}",
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH_TRAIN,
        per_device_eval_batch_size=BATCH_EVAL,
        learning_rate=LR,
        weight_decay=WEIGHT_DECAY,
        warmup_ratio=WARMUP_RATIO,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="f1_negative",
        greater_is_better=True,
        fp16=USE_GPU,
        seed=SEED,
        report_to="none",
    )


def compute_hf_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "f1_negative": f1_score(labels, preds, labels=[NEG_IDX], average="macro"),
        "f1_macro":    f1_score(labels, preds, average="macro"),
    }


def run_distilbert_experiment(run_name: str, train_dataset, trainer_class=Trainer,
                              extra_params: dict = None):
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_CHECKPOINT, num_labels=3,
        id2label=ID2LABEL, label2id=LABEL2ID,
    )
    args = get_training_args(run_name)
    trainer = trainer_class(
        model=model, args=args,
        train_dataset=train_dataset,
        eval_dataset=val_ds,
        compute_metrics=compute_hf_metrics,
    )
    with mlflow.start_run(run_name=run_name):
        params = {"model": MODEL_CHECKPOINT, "epochs": EPOCHS, "lr": LR,
                  "max_length": MAX_LENGTH, "batch_size": BATCH_TRAIN}
        if extra_params:
            params.update(extra_params)
        mlflow.log_params(params)

        trainer.train()

        for split_name, ds, y_true in [
            ("val",  val_ds,  y_val),
            ("test", test_ds, y_test),
            ("oot",  oot_ds,  y_oot),
        ]:
            preds = np.argmax(trainer.predict(ds).predictions, axis=-1)
            evaluate(y_true, preds, split_name)

    return trainer


baseline_trainer = run_distilbert_experiment(
    "distilbert-baseline",
    train_ds,
    extra_params={"class_weight": "none"},
)

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
  6%|▌         | 500/8604 [05:12<1:25:08,  1.59it/s]

{'loss': 0.5526, 'grad_norm': 1.529184341430664, 'learning_rate': 1.1567944250871081e-05, 'epoch': 0.23}


 12%|█▏        | 1000/8604 [10:25<1:19:14,  1.60it/s]

{'loss': 0.2715, 'grad_norm': 4.3347392082214355, 'learning_rate': 1.964871496835852e-05, 'epoch': 0.46}


 17%|█▋        | 1500/8604 [15:35<1:12:52,  1.62it/s]

...


── VAL ──
              precision    recall  f1-score   support

    negative       0.90      0.90      0.90       910
     neutral       0.51      0.48      0.50       310
    positive       0.97      0.97      0.97      3082

    accuracy                           0.92      4302
   macro avg       0.79      0.78      0.79      4302
weighted avg       0.92      0.92      0.92      4302



100%|██████████| 135/135 [00:38<00:00,  3.53it/s]



── TEST ──
              precision    recall  f1-score   support

    negative       0.90      0.91      0.90       911
     neutral       0.52      0.52      0.52       309
    positive       0.97      0.97      0.97      3082

    accuracy                           0.92      4302
   macro avg       0.80      0.80      0.80      4302
weighted avg       0.92      0.92      0.92      4302



100%|██████████| 161/161 [00:41<00:00,  3.84it/s]


── OOT ──
              precision    recall  f1-score   support

    negative       0.89      0.89      0.89      1119
     neutral       0.53      0.51      0.52       407
    positive       0.97      0.97      0.97      3621

    accuracy                           0.92      5147
   macro avg       0.80      0.79      0.79      5147
weighted avg       0.92      0.92      0.92      5147



## [4] Experiment B — Weighted loss

Inverse-frequency class weights in the cross-entropy loss — upweights negative and
(especially) neutral without changing the data.

In [ ]:
from torch import nn

total = len(train_df)
n_neg = (train_df.label == "negative").sum()
n_neu = (train_df.label == "neutral").sum()
n_pos = (train_df.label == "positive").sum()

class_weights = torch.tensor([
    total / (3 * n_neg),   # negative — upweighted
    total / (3 * n_neu),   # neutral  — most upweighted
    total / (3 * n_pos),   # positive — downweighted
], dtype=torch.float)

print("Class weights:", class_weights.tolist())


class WeightedLossTrainer(Trainer):
    # **kwargs absorbs num_items_in_batch on newer transformers versions
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        loss = nn.CrossEntropyLoss(
            weight=class_weights.to(model.device)
        )(outputs.logits, labels)
        return (loss, outputs) if return_outputs else loss


weighted_trainer = run_distilbert_experiment(
    "distilbert-weighted-loss",
    train_ds,
    trainer_class=WeightedLossTrainer,
    extra_params={"class_weight": "inverse_freq"},
)

Class weights: [1.5748214721679688, 4.638495922088623, 0.4652417302131653]


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
  1%|          | 106/8604 [01:03<1:24:18,  1.68it/s]

## [5] Experiment C — Weighted loss + oversample neutral

Neutral upsampled to 8K rows in **training data only** before tokenizing.

In [ ]:
neutral_rows = train_df[train_df.label == "neutral"]
neutral_up = resample(neutral_rows, replace=True, n_samples=8000, random_state=SEED)
train_bal_df = pd.concat([train_df[train_df.label != "neutral"], neutral_up]
                         ).sample(frac=1, random_state=SEED)

print("Balanced train label counts:", train_bal_df.label.value_counts().to_dict())

train_ds_bal = encode_df(train_bal_df)

oversample_trainer = run_distilbert_experiment(
    "distilbert-weighted-loss-oversample",
    train_ds_bal,
    trainer_class=WeightedLossTrainer,
    extra_params={"class_weight": "inverse_freq", "neutral_oversample_n": 8000},
)

## [6] Threshold tuning on best DistilBERT model

Compare `val_f1_negative` across Experiments A–C in the MLflow UI, set `best_trainer`
below to the winner, then tune the negative-class threshold on **val** only.

In [ ]:
# Replace with whichever trainer won on val_f1_negative in MLflow
best_trainer = weighted_trainer   # update after reviewing MLflow

# Softmax probabilities on the val set
val_logits = best_trainer.predict(val_ds).predictions
val_probs = torch.softmax(torch.tensor(val_logits), dim=-1).numpy()
val_probs_neg = val_probs[:, NEG_IDX]
val_base_preds = np.argmax(val_probs, axis=-1)

thresholds = np.arange(0.10, 0.70, 0.02)
results = []
for t in thresholds:
    preds = np.where(val_probs_neg >= t, NEG_IDX, val_base_preds)
    results.append({
        "threshold":          t,
        "f1_negative":        f1_score(y_val, preds, labels=[NEG_IDX], average="macro"),
        "recall_negative":    recall_score(y_val, preds, labels=[NEG_IDX], average="macro"),
        "precision_negative": precision_score(y_val, preds, labels=[NEG_IDX], average="macro"),
    })

results_df = pd.DataFrame(results)
best_threshold = results_df.loc[results_df.f1_negative.idxmax(), "threshold"]
print(f"Best threshold: {best_threshold:.2f}")

plt.figure(figsize=(9, 4))
plt.plot(results_df.threshold, results_df.f1_negative,        label="F1-negative")
plt.plot(results_df.threshold, results_df.recall_negative,    label="Recall-negative")
plt.plot(results_df.threshold, results_df.precision_negative, label="Precision-negative")
plt.axvline(best_threshold, color="red", linestyle="--", label=f"Best t={best_threshold:.2f}")
plt.xlabel("Negative class threshold")
plt.legend()
plt.title("Threshold tuning — DistilBERT")
plt.tight_layout()
threshold_plot_path = ROOT / "notebooks" / "distilbert_threshold_curve.png"
plt.savefig(threshold_plot_path)
plt.show()

## [7] Final evaluation on test + OOT

Best trainer + best threshold. These numbers go in the report and into the
baseline-vs-DistilBERT comparison.

In [ ]:
def predict_distilbert_with_threshold(trainer, dataset, threshold):
    logits = trainer.predict(dataset).predictions
    probs = torch.softmax(torch.tensor(logits), dim=-1).numpy()
    base_pred = np.argmax(probs, axis=-1)
    return np.where(probs[:, NEG_IDX] >= threshold, NEG_IDX, base_pred)


with mlflow.start_run(run_name="distilbert-final"):
    mlflow.log_params({"neg_threshold": best_threshold})
    mlflow.log_artifact(str(threshold_plot_path))

    evaluate(y_test, predict_distilbert_with_threshold(best_trainer, test_ds, best_threshold), "test")
    evaluate(y_oot,  predict_distilbert_with_threshold(best_trainer, oot_ds,  best_threshold), "oot")

    final_run_id = mlflow.active_run().info.run_id

## [8] Register best model to MLflow

Saves model + tokenizer to one directory (so the API can rehydrate both) and registers as
**`sentiment-distilbert`**. Promote to `Staging` manually in the MLflow UI before the
FastAPI shadow deploy can load it.

In [ ]:
best_dir = ARTIFACT_DIR / "distilbert_best"
best_trainer.save_model(str(best_dir))
tokenizer.save_pretrained(str(best_dir))
print(f"Saved model + tokenizer: {best_dir}")

with mlflow.start_run(run_id=final_run_id):
    mlflow.pytorch.log_model(
        best_trainer.model,
        artifact_path="model",
        registered_model_name="sentiment-distilbert",
    )
print("Registered: sentiment-distilbert")

## [9] LR sensitivity experiment (optional, recommended for the report)

Three single-epoch probes to find the best learning rate before committing to full
training. Each probe is logged to MLflow as `distilbert-lr-probe-{lr}`.

In [ ]:
for lr in [5e-6, 2e-5, 5e-5]:
    args = get_training_args(f"distilbert-lr-{lr}")
    args.num_train_epochs = 1   # probe only
    args.learning_rate = lr

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_CHECKPOINT, num_labels=3, id2label=ID2LABEL, label2id=LABEL2ID
    )
    trainer = WeightedLossTrainer(
        model=model, args=args,
        train_dataset=train_ds, eval_dataset=val_ds,
        compute_metrics=compute_hf_metrics,
    )
    with mlflow.start_run(run_name=f"distilbert-lr-probe-{lr}"):
        mlflow.log_param("lr", lr)
        trainer.train()
        val_preds = np.argmax(trainer.predict(val_ds).predictions, axis=-1)
        evaluate(y_val, val_preds, "val")

## Model selection rule (from the instructions)

Pick the run with the highest **`test_f1_negative`** across both notebooks.
If two runs are within 0.01 of each other, prefer the **simpler** model
(logreg over distilbert; no sampling over sampling).

Registered names: `sentiment-baseline` (notebook 01), `sentiment-distilbert` (this notebook).
Both must be promoted to `Staging` in the MLflow UI before the FastAPI shadow deploy
can load them.